# Part 3 · Generative CSP of T2  (~12 min)

T2 is a triptycene tris-benzimidazolone. Pulido *et al.* (*Nature* **543**, 657, 2017) predicted several porous,
low-density forms for it with CSP, and some of them were then made. Here we ask Clari-H for 100 crystal structures
of T2 (Z = 2, 4 or 6), starting from the SMILES only, and check them against five known structures.

In [ ]:
# @title 0 · Setup  (run this cell first; ~1-2 min on Colab)
import os, sys
REPO = "https://github.com/fmggggg/agentic-ai-for-functional-materials-discovery"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/csp-tutorial"):
        !git clone -q {REPO} /content/csp-tutorial
    %cd /content/csp-tutorial
    !pip -q install pyxtal mace-torch py3Dmol pymatgen rdkit oxtalign average-minimum-distance "requests==2.32.4"
import csp_utils as cu
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("ready")

## 3.1 T2 and the five reference structures

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
display(Draw.MolToImage(Chem.MolFromSmiles(cu.T2_SMILES), size=(320, 220)))
refs = {r: cu.t2_reference(r) for r in cu.T2_REFS}
ref_e = pd.read_csv(cu.T2 / "references.csv").set_index("reference")
pd.DataFrame({r: {"Z": int(round(len(s) / 46)), "density (g/cm3)": round(s.density, 3),
                  "E in 2017 study (kJ/mol)": ref_e.loc[r, "original_csp_energy_kjmol"]} for r, s in refs.items()}).T

In [ ]:
cu.show(cu.to_ase(refs["gamma"]), supercell=(2, 2, 2))   # gamma, looking down the channels

> **Question 6.** γ has less than a third of the density of the global minimum and sits about 60 kJ/mol higher in energy.
> Why would anyone want it?

## 3.2 Score the raw samples (~1 min)

The samples come straight from the model, with no relaxation. OXtalign overlays a cluster of 15 molecules
from each sample on the reference and reports how many molecules match and their RMSD.

In [ ]:
import time
REF = "gamma"                      # or beta, delta, csp_global_min, alpha
names = cu.t2_raw_names()          # the 100 samples
t0 = time.time()
S = cu.oxtalign_scores(names, refs[REF])
print(f"{len(names)} samples scored against {REF} in {time.time() - t0:.0f} s;",
      (S["molecules matched (of 15)"] == 15).sum(), "match all 15 molecules")
S.head(8)

> **Question 7.** Some samples already match γ before any relaxation. Do we need to relax all 100 to find them?

## 3.3 Relax the best three (~2 min)

In [ ]:
E = cu.t2_endpoints().set_index("name")
fmt = lambda res: f"{res[1]}/15, {res[0]:.2f} Å"
rows = []
for name in S["sample"][:3]:
    raw = cu.t2_raw_structure(name)
    t0 = time.time()
    relaxed = cu.relax(cu.to_ase(raw), fmax=0.1, steps=60, symmetry=False)
    n_mol = len(relaxed) // 46
    row = {"sample": name, "raw": fmt(cu.compack_like(refs[REF], raw)),
           "60 steps here": fmt(cu.compack_like(refs[REF], cu.to_pmg(relaxed))), "seconds": round(time.time() - t0),
           "E, 60 steps (kJ/mol)": round(cu.energy_per_molecule(relaxed, n_mol) - cu.T2_E0_KJMOL, 1)}
    if E.loc[name, "converged"]:
        row["full relaxation (GPU)"] = fmt(cu.compack_like(refs[REF], cu.t2_structure(name)))
        row["E, full (kJ/mol)"] = round(E.loc[name, "rel_energy_kjmol"], 1)
    rows.append(row)
pd.DataFrame(rows, dtype=object)

> **Question 8.** How much did relaxation change these structures? When would you still relax every sample?

## 3.4 All 100 samples on an energy–density map

Before the school every sample was relaxed on a GPU with the same settings (≤ 500 steps).
Samples with clashing atoms were skipped. Orange: samples that matched γ in 3.2.

In [ ]:
import json
f = json.loads((cu.T2 / "funnel.json").read_text())
print(f"{f['sampled']} sampled, {f['relaxed']} relaxed, {f['converged']} converged")
C = cu.t2_endpoints()
C = C[C.converged]
hit = C.name.isin(S.loc[S["molecules matched (of 15)"] == 15, "sample"])
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(C.density, C.rel_energy_kjmol, s=14, c="#BDBDBD", label="relaxed samples")
ax.scatter(C.density[hit], C.rel_energy_kjmol[hit], s=40, c="#E0A526", edgecolor="k", label=f"matched {REF} before relaxation")
for r, s in refs.items():
    ax.axvline(s.density, c="k", ls=":", lw=0.8); ax.text(s.density, 125, r, rotation=90, fontsize=8, ha="right", va="top")
ax.set_ylim(-5, 130); ax.set_xlabel("density (g/cm$^3$)"); ax.set_ylabel("relative MACE-OFF energy (kJ/mol per T2)")
ax.legend(fontsize=8);

> **Question 9.** Where are the γ-like samples compared with other structures of similar density?
> Pulido *et al.* looked for exactly this: structures that are low in energy for their density.

## 3.5 Is it the same crystal?

pymatgen `StructureMatcher` compares unit cells and gives a normalised RMSD (not Å). OXtalign compares clusters of
15 molecules, like CCDC COMPACK. AMD fingerprints need no alignment and are fast enough to deduplicate.

In [ ]:
best = S["sample"][0]
s = cu.t2_structure(best)
print("pymatgen RMSD (normalised):", cu.pymatgen_rmsd(s, refs[REF]))
print("OXtalign (RMSD15 in Å, matched, status):", cu.compack_like(refs[REF], s))

G = [n for n in C.name[hit]]                  # relaxed γ-like samples
fps = [cu.amd_fingerprint(cu.t2_structure(n), k=100) for n in G]
for tol in (0.01, 0.05, 0.1):
    print(f"AMD tolerance {tol} Å: {len(set(cu.group_duplicates(fps, tol)))} distinct structures among {len(G)}")

> **Question 10.** How many different γ-like structures did the model find? Why does the answer depend on the tolerance,
> and what would you report in a paper?

## 3.6 (Optional) Run Clari yourself

Needs a GPU: `Runtime → Change runtime type → T4 GPU`, then set `RUN_CLARI = True`. The weights are downloaded on first use (0.3–1.4 GB).

In [ ]:
RUN_CLARI = False
SMILES = "CC(=O)Nc1ccc(O)cc1"   # paracetamol
if RUN_CLARI:
    !pip -q install clari
    from clari.inference import ClariSampler
    crystals = ClariSampler("clari-m").sample(SMILES, id="my_molecule", samples=8, copies=4)
    form1 = cu.to_pmg(cu.load_cif(cu.DATA / "paracetamol_form_I.cif"))
    res = [cu.compack_like(form1, cu.to_pmg(c.to_ase())) for c in crystals]
    display(pd.DataFrame(res, columns=["RMSD15 (Å)", "matched", "status"]))
    best = max(range(len(res)), key=lambda i: (res[i][1], -(res[i][0] or 9)))
    r = cu.relax(crystals[best].to_ase(), fmax=0.1, steps=100, symmetry=False)
    print("best sample after relaxation vs form I:", cu.compack_like(form1, cu.to_pmg(r)))
    cu.show(r, supercell=(2, 2, 2))

---
Further reading: Pulido *et al.*, *Nature* **543**, 657 (2017) · OXtalign: github.com/OXtal/oxtalign ·
AMD: `pip install average-minimum-distance`